# Download antiobiotics data from chEMBL

In [32]:
import os
from chembl_webresource_client.new_client import new_client
import pandas as pd
import time
from tqdm import tqdm

In [33]:
# Base directory path
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'

# Construct full paths for subdirectories 
Bacillus_subtilis_dataDir = os.path.join(dataDir, 'Bacillus_subtilis/')
Bacillus_cereus_dataDir = os.path.join(dataDir, 'Bacillus_cereus/')
modelBuilding_dataDir = os.path.join(dataDir, 'modelBuildingData/')

In [34]:
# List of directories to ensure they exist
directories = [dataDir, Bacillus_subtilis_dataDir, Bacillus_cereus_dataDir, modelBuilding_dataDir]

# Loop through the list and create each directory if it doesn't exist
for path in directories:
    try:
        os.makedirs(path, exist_ok=True)
        print(f"Directory '{path}' is ready.")
    except OSError as error:
        print(f"Error creating directory '{path}': {error}")

Directory '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/' is ready.
Directory '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Bacillus_subtilis/' is ready.
Directory '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Bacillus_cereus/' is ready.
Directory '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/' is ready.


In [35]:
# pip install chembl_webresource_client pandas tqdm requests
from chembl_webresource_client.new_client import new_client
from chembl_webresource_client.http_errors import BaseHttpException
import pandas as pd
import requests, os, time, sys
from typing import Callable, Iterable, List, Dict

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

try:
    from tqdm import tqdm   # plain text tqdm (no Jupyter widget dependency)
    useTqdm = True
except Exception:
    useTqdm = False

def _printProgress(label: str, n: int, start: float):
    dt = time.time() - start
    rate = n / (dt + 1e-9)
    print(f"{label}: {n:,} records … {rate:.1f} rec/s", flush=True)

def _consumeClient(factory: Callable[[], Iterable[dict]], label: str, maxRetries: int = 5, backoff: float = 1.5) -> List[dict]:
    """
    Consume a chembl_webresource_client iterable with retries (500s, etc.).
    We pass a factory() that recreates the iterable fresh after each error.
    """
    attempt = 0
    while True:
        items, n = [], 0
        start = time.time()
        if useTqdm: bar = tqdm(unit="rec", desc=label, leave=False)
        try:
            for rec in factory():
                items.append(rec); n += 1
                if useTqdm: bar.update(1)
                elif n % 1000 == 0: _printProgress(label, n, start)
            if useTqdm: bar.close()
            _printProgress(f"{label} done", n, start)
            return items
        except BaseHttpException as e:
            if useTqdm: bar.close()
            attempt += 1
            if attempt > maxRetries:
                raise
            sleepFor = (backoff ** (attempt - 1)) + 0.25 * attempt
            print(f"[WARN] {label}: HTTP error ({type(e).__name__}). Retry {attempt}/{maxRetries} in {sleepFor:.1f}s…",
                  file=sys.stderr, flush=True)
            time.sleep(sleepFor)

def _fetchAllREST(endpoint: str, params: dict, pageSize: int = 1000, maxRetries: int = 5) -> List[Dict]:
    """
    Requests-based fallback for ChEMBL /data endpoints (handles JSON pagination).
    """
    params = dict(params or {})
    params.update({"format": "json", "limit": pageSize, "offset": 0})
    allRows: List[Dict] = []
    offset = 0
    while True:
        params["offset"] = offset
        for attempt in range(1, maxRetries + 1):
            try:
                r = requests.get(f"{BASE_URL}/{endpoint}", params=params, timeout=120)
                if r.status_code >= 500:
                    raise requests.HTTPError(f"Server {r.status_code}")
                r.raise_for_status()
                payload = r.json()
                break
            except requests.RequestException as e:
                if attempt == maxRetries: raise
                sleepFor = (1.5 ** (attempt - 1)) + 0.25 * attempt
                print(f"[WARN] REST {endpoint}: {e}. Retry {attempt}/{maxRetries} in {sleepFor:.1f}s…",
                      file=sys.stderr, flush=True)
                time.sleep(sleepFor)
        items = payload.get("items") or payload.get(endpoint) or []
        allRows.extend(items)
        meta = payload.get("page_meta", {})
        total = meta.get("total_count"); limit = meta.get("limit", pageSize); offset = meta.get("offset", offset) + limit
        if total is None:
            if len(items) < pageSize: break
        else:
            if offset >= total: break
    return allRows

def pullWithClient(targetId: str, useRestFallback: bool = True):
    activity = new_client.activity
    assay = new_client.assay
    molecule = new_client.molecule

    # --- ACTIVITIES ---
    def activitiesFactory():
        return activity.filter(target_chembl_id=targetId).only(
            ["activity_id","assay_chembl_id","molecule_chembl_id","target_chembl_id",
             "standard_type","standard_relation","standard_value","standard_units","pchembl_value"]
        )
    try:
        activitiesList = _consumeClient(activitiesFactory, label="Activities")
    except BaseHttpException:
        if not useRestFallback: raise
        print("[INFO] Falling back to REST for Activities…")
        activitiesList = _fetchAllREST("activity", {"target_chembl_id": targetId})
    activitiesDF = pd.DataFrame(activitiesList)

    # --- ASSAYS ---
    def assaysFactory():
        return assay.filter(target_chembl_id=targetId).only(
            ["assay_chembl_id","assay_type","bao_label","description","target_chembl_id",
             "confidence_score","relationship_type","relationship_description"]
        )
    try:
        assaysList = _consumeClient(assaysFactory, label="Assays")
    except BaseHttpException:
        if not useRestFallback: raise
        print("[INFO] Falling back to REST for Assays…")
        assaysList = _fetchAllREST("assay", {"target_chembl_id": targetId})
    assaysDF = pd.DataFrame(assaysList)

    # --- MOLECULES (chunked) ---
    molIds = activitiesDF["molecule_chembl_id"].dropna().unique().tolist() if not activitiesDF.empty else []
    moleculesDF = pd.DataFrame()
    if molIds:
        chunks = [molIds[i:i+200] for i in range(0, len(molIds), 200)]
        print(f"Molecules: {len(molIds):,} unique IDs across {len(chunks)} chunks (≤200 each)")
        frames = []
        if useTqdm: chunkBar = tqdm(total=len(chunks), desc="Molecule chunks", unit="chunk", leave=False)
        for idx, chunk in enumerate(chunks, start=1):
            idsStr = ",".join(chunk)
            def molFactory():
                return molecule.filter(molecule_chembl_id__in=idsStr)
            try:
                partList = _consumeClient(molFactory, label=f"Chunk {idx}/{len(chunks)}")
            except BaseHttpException:
                if not useRestFallback: raise
                print(f"[INFO] Falling back to REST for molecule chunk {idx}/{len(chunks)}…")
                partList = _fetchAllREST("molecule", {"molecule_chembl_id__in": idsStr})
            frames.append(pd.DataFrame(partList))
            if useTqdm: chunkBar.update(1)
        if useTqdm: chunkBar.close()
        moleculesDF = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    return activitiesDF, assaysDF, moleculesDF


if __name__ == "__main__":
    targetId = "CHEMBL613070"
    targetName = "Bacillus_cereus"

    targetDataDir = os.path.join(dataDir, targetName)
    os.makedirs(targetDataDir, exist_ok=True)

    print(f"Starting ChEMBL pull for {targetName} ({targetId})")
    tStart = time.time()
    activitiesDF, assaysDF, moleculesDF = pullWithClient(targetId, useRestFallback=True)

    activitiesDF.to_csv(os.path.join(targetDataDir, f"{targetName}_bioactivity.csv"), index=False)
    assaysDF.to_csv(os.path.join(targetDataDir, f"{targetName}_bioassays.csv"), index=False)
    moleculesDF.to_csv(os.path.join(targetDataDir, f"{targetName}_compounds.csv"), index=False)
    print(f"Totals — activities: {len(activitiesDF):,}, assays: {len(assaysDF):,}, molecules: {len(moleculesDF):,}")
    print(f"Elapsed: {time.time() - tStart:.1f}s")


Starting ChEMBL pull for Bacillus_cereus (CHEMBL613070)




Activities: 0rec [00:00, ?rec/s]

Activities: 1rec [00:05,  5.97s/rec]

Activities: 21rec [00:06,  4.01rec/s]

Activities: 41rec [00:08,  7.19rec/s]

Activities: 60rec [00:18,  7.19rec/s]

Activities: 61rec [00:30,  1.69rec/s]

Activities: 81rec [01:14,  1.22s/rec]

Activities: 101rec [01:56,  1.53s/rec]

Activities: 121rec [02:38,  1.73s/rec]

Activities: 141rec [03:24,  1.91s/rec]

Activities: 161rec [04:09,  2.02s/rec]

Activities: 181rec [04:53,  2.08s/rec]

Activities: 201rec [05:09,  1.69s/rec]

Activities: 221rec [05:56,  1.88s/rec]

Activities: 241rec [06:37,  1.94s/rec]

Activities: 261rec [07:20,  2.00s/rec]

Activities: 281rec [08:05,  2.07s/rec]

Activities: 301rec [08:45,  2.06s/rec]

Activities: 321rec [09:05,  1.74s/rec]

Activities: 341rec [09:46,  1.83s/rec]

Activities: 361rec [10:26,  1.88s/rec]

Activities: 381rec [11:07,  1.92s/rec]

Activities: 401rec [11:12,  1.43s/rec]

Activities: 420rec [11:28,  1.43s/rec]

Activities: 421rec [11:53,  1.61s/rec]

Activities:

Activities done: 7,696 records … 32.1 rec/s




Assays: 0rec [00:00, ?rec/s]

Assays: 1rec [00:02,  2.20s/rec]

Assays: 21rec [00:02,  9.04rec/s]

Assays: 41rec [00:03, 14.85rec/s]

Assays: 61rec [00:04, 19.04rec/s]

Assays: 81rec [00:05, 22.30rec/s]

Assays: 101rec [00:05, 23.46rec/s]

Assays: 121rec [00:06, 23.47rec/s]

Assays: 141rec [00:07, 25.70rec/s]

Assays: 161rec [00:07, 27.16rec/s]

Assays: 181rec [00:08, 27.91rec/s]

Assays: 201rec [00:09, 26.19rec/s]

Assays: 221rec [00:10, 27.52rec/s]

Assays: 241rec [00:10, 28.83rec/s]

Assays: 261rec [00:11, 28.35rec/s]

Assays: 281rec [00:12, 27.21rec/s]

Assays: 301rec [00:12, 28.01rec/s]

Assays: 321rec [00:13, 28.95rec/s]

Assays: 341rec [00:14, 29.02rec/s]

Assays: 361rec [00:15, 25.61rec/s]

Assays: 381rec [00:15, 27.05rec/s]

Assays: 401rec [00:16, 25.88rec/s]

Assays: 421rec [00:17, 26.77rec/s]

Assays: 441rec [00:18, 27.37rec/s]

Assays: 461rec [00:18, 28.25rec/s]

Assays: 481rec [00:19, 28.62rec/s]

Assays: 501rec [00:20, 28.03rec/s]

Assays: 521rec [00:20, 28.17rec/s]

As

Assays done: 870 records … 26.7 rec/s
Molecules: 5,229 unique IDs across 27 chunks (≤200 each)


...

Chunk 26/27 done: 200 records … 14.7 rec/s




Molecule chunks:  96%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎    | 26/27 [05:05<00:12, 12.66s/chunk]


Chunk 27/27: 0rec [00:00, ?rec/s]


Chunk 27/27: 1rec [00:01,  1.42s/rec]


Chunk 27/27: 21rec [00:02,  9.83rec/s]


                                      

Chunk 27/27 done: 29 records … 11.4 rec/s




Molecule chunks: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 27/27 [05:07<00:00,  9.62s/chunk]

                                                                                                                                                                                       /tmp/ipykernel_2421905/221669956.py:137: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  moleculesDF = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


Totals — activities: 7,696, assays: 870, molecules: 5,229
Elapsed: 5604.6s


### Alternative method fetched from chEMBL API documentation

https://www.ebi.ac.uk/training/online/courses/embl-ebi-programmatically/wp-content/uploads/sites/128/2020/08/ChEMBL-programmatically.pdf

In [36]:
import os
import time
import math
import json
from typing import Dict, List, Tuple, Optional
import requests
import pandas as pd

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"  # live docs mentioned in slides
MAX_LIMIT = 1000                                    # per pagination rules
TIMEOUT = 120

# ---- Helpers ----------------------------------------------------------------

def _should_use_post(params: Dict) -> bool:
    """
    If the encoded query would be too long (lots of IDs in __in, etc.),
    use POST with X-HTTP-Method-Override: GET as suggested by the slides.
    """
    # crude heuristic: if any value is > 1500 chars, play safe and POST
    for v in params.values():
        if isinstance(v, str) and len(v) > 1500:
            return True
    return False

def _request_page(endpoint: str, params: Dict, method_override_post: bool = False) -> Dict:
    """
    Fetch a single page. Use GET normally; if method_override_post=True,
    use POST with 'X-HTTP-Method-Override: GET' (per training doc).
    Retries on 5xx / network errors with exponential backoff.
    """
    url = f"{BASE_URL}/{endpoint}"
    headers = {}
    tries, max_tries = 0, 6
    while True:
        tries += 1
        try:
            if method_override_post:
                headers = {
                    "X-HTTP-Method-Override": "GET",
                    "Content-Type": "application/json",
                }
                # NB: For method-override, send params as JSON body
                r = requests.post(url, data=json.dumps(params), headers=headers, timeout=TIMEOUT)
            else:
                r = requests.get(url, params=params, timeout=TIMEOUT)

            # Retry on 5xx transparently
            if r.status_code >= 500:
                raise requests.HTTPError(f"Server {r.status_code}: {r.text[:200]}")

            r.raise_for_status()
            return r.json() if params.get("format") == "json" else r.content
        except (requests.Timeout, requests.ConnectionError, requests.HTTPError) as e:
            if tries >= max_tries:
                raise
            sleep_for = (1.5 ** (tries - 1)) + 0.25 * tries
            print(f"[WARN] {endpoint}: {e}. Retry {tries}/{max_tries} in {sleep_for:.1f}s…", flush=True)
            time.sleep(sleep_for)

def fetch_all(endpoint: str, filters: Optional[Dict] = None, page_size: int = MAX_LIMIT) -> List[Dict]:
    """
    Generic paginator for ChEMBL /data endpoints (JSON).
    - endpoint: e.g., 'activity', 'assay', 'molecule'
    - filters: dict of filter params (Django-style operators supported, per slides)
    Returns a flat list of item dicts.
    """
    params = dict(filters or {})
    params.update({"format": "json", "limit": min(page_size, MAX_LIMIT), "offset": 0})
    all_items: List[Dict] = []
    offset = 0

    # Decide once if we should POST-override (huge param values)
    method_override_post = _should_use_post(params)

    while True:
        params["offset"] = offset
        payload = _request_page(endpoint, params, method_override_post=method_override_post)
        items = payload.get("items") or payload.get(endpoint) or []
        all_items.extend(items)

        meta = payload.get("page_meta", {})
        total = meta.get("total_count")
        limit = meta.get("limit", params["limit"])
        cur_offset = meta.get("offset", offset)

        # Progress prints every ~5k rows
        if len(all_items) % 5000 == 0 and len(all_items) > 0:
            print(f"{endpoint}: {len(all_items):,} records…", flush=True)

        if total is None:
            # No meta? then stop when short page
            if len(items) < limit:
                break
            offset = cur_offset + limit
        else:
            offset = cur_offset + limit
            if offset >= total:
                break

    print(f"{endpoint}: done — {len(all_items):,} records", flush=True)
    return all_items

def chunk_list(seq: List[str], n: int) -> List[List[str]]:
    return [seq[i:i+n] for i in range(0, len(seq), n)]

# ---- Pipeline ---------------------------------------------------------------

def pull_target_to_csv(
    target_id: str,
    out_dir: str,
    out_prefix: Optional[str] = None,
    filter_activity: Optional[Dict] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Pull Activities, Assays, and Molecules for a given target_chembl_id and save CSVs.
    - filter_activity allows narrowing (e.g., {'pchembl_value__gte': 6, 'standard_type': 'IC50'})
    """
    os.makedirs(out_dir, exist_ok=True)
    prefix = out_prefix or target_id

    # ---- Activities ----
    base_filters = {"target_chembl_id": target_id}
    if filter_activity:
        base_filters.update(filter_activity)
    activities = fetch_all("activity", base_filters)
    activitiesDF = pd.DataFrame(activities)
    activities_csv = os.path.join(out_dir, f"{prefix}_activities.csv")
    activitiesDF.to_csv(activities_csv, index=False)
    print(f"Saved: {activities_csv}")

    # ---- Assays ----
    assays = fetch_all("assay", {"target_chembl_id": target_id})
    assaysDF = pd.DataFrame(assays)
    assays_csv = os.path.join(out_dir, f"{prefix}_assays.csv")
    assaysDF.to_csv(assays_csv, index=False)
    print(f"Saved: {assays_csv}")

    # ---- Molecules (for IDs appearing in activities) ----
    moleculesDF = pd.DataFrame()
    if not activitiesDF.empty and "molecule_chembl_id" in activitiesDF.columns:
        mol_ids = (
            activitiesDF["molecule_chembl_id"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )
        print(f"Molecules: {len(mol_ids):,} unique molecule IDs from activities")

        # Query in chunks with __in; switch to POST override automatically for long lists
        frames = []
        for i, chunk in enumerate(chunk_list(mol_ids, 200), start=1):
            ids_param = ",".join(chunk)
            rows = fetch_all("molecule", {"molecule_chembl_id__in": ids_param})
            frames.append(pd.DataFrame(rows))
            print(f" - chunk {i}: {len(rows):,} rows")

        moleculesDF = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

        # Optional: flatten nested fields for convenience
        if not moleculesDF.empty and "molecule_properties" in moleculesDF.columns:
            props = pd.json_normalize(moleculesDF["molecule_properties"])
            props.columns = [f"props__{c}" for c in props.columns]
            moleculesDF = pd.concat([moleculesDF.drop(columns=["molecule_properties"]), props], axis=1)
        if not moleculesDF.empty and "molecule_structures" in moleculesDF.columns:
            structs = pd.json_normalize(moleculesDF["molecule_structures"])
            structs.columns = [f"structs__{c}" for c in structs.columns]
            moleculesDF = pd.concat([moleculesDF.drop(columns=["molecule_structures"]), structs], axis=1)

    molecules_csv = os.path.join(out_dir, f"{prefix}_molecules.csv")
    moleculesDF.to_csv(molecules_csv, index=False)
    print(f"Saved: {molecules_csv}")

    return activitiesDF, assaysDF, moleculesDF

# ---- Run --------------------------------------------------------------------

if __name__ == "__main__":
    # Example: Bacillus cereus (replace with any Target ChEMBL ID)
    targetId = "CHEMBL613070"
    targetName = "Bacillus_cereus"
    outDir = os.path.expanduser(f"~/chemblData/{targetName}")

    # Optional activity filter examples (uncomment to narrow payload):
    # filterActivity = {"pchembl_value__gte": 6}
    # filterActivity = {"standard_type": "IC50", "pchembl_value__gte": 6}
    filterActivity = None

    print(f"Fetching ChEMBL data for {targetName} ({targetId}) → {outDir}")
    t0 = time.time()
    activitiesDF, assaysDF, moleculesDF = pull_target_to_csv(
        target_id=targetId,
        out_dir=outDir,
        out_prefix=targetName,
        filter_activity=filterActivity,
    )
    print(f"Done in {time.time() - t0:.1f}s")
    print(f"Totals — activities: {len(activitiesDF):,}, assays: {len(assaysDF):,}, molecules: {len(moleculesDF):,}")


Fetching ChEMBL data for Bacillus_cereus (CHEMBL613070) → /users/sghosh6/chemblData/Bacillus_cereus
activity: done — 0 records
Saved: /users/sghosh6/chemblData/Bacillus_cereus/Bacillus_cereus_activities.csv
assay: done — 0 records
Saved: /users/sghosh6/chemblData/Bacillus_cereus/Bacillus_cereus_assays.csv
Saved: /users/sghosh6/chemblData/Bacillus_cereus/Bacillus_cereus_molecules.csv
Done in 57.0s
Totals — activities: 0, assays: 0, molecules: 0


### working with manually downloaded data from chEMBL

In [37]:
import os
import zipfile
import pandas as pd
import glob
import numpy as np
import warnings
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import re

In [39]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData')

### The code below will process the data based on the bacteria name user provided

In [48]:
import pandas as pd
import numpy as np
import os
import csv
import re
import matplotlib.pyplot as plt
from io import StringIO

def readChemblCsvFile(filePath):
    """
    Read ChEMBL CSV file with optimized settings for large files.
    """
    print(f"Reading ChEMBL CSV file: {filePath}")
    if not os.path.exists(filePath):
        print(f"Error: File not found at {filePath}")
        return None
    
    try:
        # Define dtypes upfront to reduce memory and increase speed
        dtype_dict = {
            'Standard Value': 'float32',
            'pChEMBL Value': 'float32',
            'Molecular Weight': 'float32',
            'AlogP': 'float32',
            'Standard Type': 'category',
            'Standard Units': 'category',
            'Standard Relation': 'category',
            'Target Organism': 'category'
        }
        
        dataFrame = pd.read_csv(
            filePath,
            sep=';',
            quotechar='"',
            quoting=csv.QUOTE_ALL,
            doublequote=True,
            skipinitialspace=True,
            on_bad_lines='skip',
            encoding='utf-8',
            dtype=dtype_dict,
            engine='c',  # Faster C engine
            na_values=['', 'NA', 'N/A', 'null']
        )
        return dataFrame
    except Exception as e:
        print(f"Error reading file: {e}")
        return None


def processBacteriaData(bacteriaName):
    """
    Optimized main function to process bioactivity and compound data.
    """
    
    if not os.path.exists(bacteriaDataDir):
        print(f"Error: Data directory for '{bacteriaName}' not found at {bacteriaDataDir}")
        return

    # ### ChEMBL Bioassay Data
    print("\n--- Processing Bioassay Data ---")
    bioactivityFile = os.path.join(bacteriaDataDir, f'{bacteriaName}_bioactivities.csv')
    chemblData = readChemblCsvFile(bioactivityFile)

    if chemblData is None:
        return

    print(f"Dataset loaded successfully with Shape: {chemblData.shape}")

    # #### Keep only relevant properties - avoid .copy() if not modifying
    selectedColumns = ['Smiles', 'Standard Value', 'Standard Units', 'Standard Type',
                      'Standard Relation', 'pChEMBL Value', 'Molecule ChEMBL ID',
                      'Molecular Weight', 'AlogP', 'Target ChEMBL ID',
                      'Target Name', 'Target Organism']

    chemblDataCleaned = chemblData[[col for col in selectedColumns if col in chemblData.columns]]
    print(f"Dataset shape after selecting columns: {chemblData.shape} → {chemblDataCleaned.shape}")
    
    outputPath = os.path.join(modelBuildingDataDir, f'{bacteriaName}Bioassay_chEMBL_cleaned.csv')
    chemblDataCleaned.to_csv(outputPath, index=False)
    print(f"Cleaned bioassay data saved to: {outputPath}")

    # ### Count Standard Type and Generate Plot
    print("\n--- Analyzing Standard Types ---")
    typeCounts = chemblDataCleaned["Standard Type"].value_counts()
    total = typeCounts.sum()
    legendLabels = [f"{stype} ({count}, {count/total:.1%})" for stype, count in typeCounts.items()]

    plt.figure(figsize=(10, 8))
    wedges, texts, autotexts = plt.pie(
        typeCounts, labels=typeCounts.index, autopct='%1.1f%%',
        startangle=140, wedgeprops={'edgecolor': 'black'}, textprops={'fontsize': 9}
    )
    plt.legend(wedges, legendLabels, title="Standard Type", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
    plt.title(f"Distribution of Standard Types for {bacteriaName}")
    plt.tight_layout()
    plotPath = os.path.join(modelBuildingDataDir, f'{bacteriaName}_standard_types.png')
    plt.savefig(plotPath, dpi=150)
    print(f"Standard types plot saved to: {plotPath}")
    plt.close()  # Close to free memory

    # ### Check for empty (NaN) values
    print("\n--- Checking for NaN values ---")
    nanCounts = chemblDataCleaned.isnull().sum()
    totalRows = len(chemblDataCleaned)
    for column, nanCount in nanCounts.items():
        percentage = (nanCount / totalRows) * 100
        print(f"{column:25s}: {nanCount:6,} ({percentage:5.1f}%)")

    # ### Count unique compounds
    if "Molecule ChEMBL ID" in chemblDataCleaned.columns:
        uniqueCount = chemblDataCleaned["Molecule ChEMBL ID"].nunique()
        print(f"\nNumber of unique compounds for {bacteriaName}: {uniqueCount}")

    # ### ChEMBL compounds data - OPTIMIZED VERSION
    print("\n--- Processing Compounds Data ---")
    compoundsFile = os.path.join(bacteriaDataDir, f'{bacteriaName}_compounds.csv')
    if not os.path.exists(compoundsFile):
        print(f"Error: Compounds file not found at {compoundsFile}")
        return

    # Try using pandas directly with error handling
    try:
        chemblCompoundData = pd.read_csv(
            compoundsFile,
            sep=';',
            quotechar='"',
            encoding='utf-8',
            on_bad_lines='skip',
            engine='c',
            low_memory=False
        )
        print("Successfully read compounds file with pandas")
    except Exception as e:
        print(f"Pandas failed, using optimized manual parser: {e}")
        chemblCompoundData = readCompoundsFileOptimized(compoundsFile)
    
    # Standardize column names efficiently
    chemblCompoundData.columns = chemblCompoundData.columns.str.replace(r"[^\w]+", "_", regex=True).str.strip("_").str.lower()
    
    # Standardize the ChEMBL ID column name for merging
    if 'molecule_chembl_id' in chemblCompoundData.columns:
        chemblCompoundData.rename(columns={'molecule_chembl_id': 'chembl_id'}, inplace=True)

    # Convert numeric columns efficiently
    numCols = ["molecular_weight", "alogp", "polar_surface_area", "hba", "hbd", 
               "ro5_violations", "rotatable_bonds", "qed_weighted", "heavy_atoms"]
    
    for c in numCols:
        if c in chemblCompoundData.columns:
            chemblCompoundData[c] = pd.to_numeric(chemblCompoundData[c], errors="coerce").astype('float32')
    
    print(f"Compounds data shape: {chemblCompoundData.shape}")
    outputPath = os.path.join(modelBuildingDataDir, f'{bacteriaName}Compounds_chEMBL_cleaned.csv')
    chemblCompoundData.to_csv(outputPath, index=False)
    print(f"Cleaned compounds data saved to: {outputPath}")

    # ### Combine Bioassay and Compound Data
    print("\n--- Combining Bioassay and Compound Data ---")
    colsToImport = ['chembl_id', 'type', 'polar_surface_area', 'hba', 'hbd', 'rotatable_bonds', 
                    'qed_weighted', 'heavy_atoms', 'aromatic_rings', 'ro5_violations', 
                    'molecular_formula', 'inchi_key', 'inchi']
    cmpdSub = chemblCompoundData[[c for c in colsToImport if c in chemblCompoundData.columns]]

    # Use merge with indicator to check merge quality
    chemblDataCombined = chemblDataCleaned.merge(
        cmpdSub, 
        how='left', 
        left_on='Molecule ChEMBL ID', 
        right_on='chembl_id',
        suffixes=('', '_cmp'),
        validate='m:1'  # Many-to-one merge validation
    )
    
    if 'chembl_id' in chemblDataCombined.columns:
        chemblDataCombined.drop(columns=['chembl_id'], inplace=True)

    # ### Rename columns efficiently
    rename_dict = {
        "Smiles": "smiles", 
        "Standard Value": "standard_value", 
        "Standard Units": "standard_units",
        "Standard Type": "standard_type", 
        "Standard Relation": "standard_relation", 
        "pChEMBL Value": "pchembl_value",
        "Molecule ChEMBL ID": "molecule_chembl_id", 
        "Molecular Weight": "molecular_weight",
    }
    chemblDataCombined.rename(columns=rename_dict, inplace=True)

    # ### Reorder columns
    desiredOrder = [
        "molecule_chembl_id", "smiles", "molecular_formula", "inchi_key", "inchi", "molecular_weight", "type",
        "standard_value", "standard_units", "standard_relation", "standard_type", "pchembl_value", "AlogP",
        "polar_surface_area", "hba", "hbd", "rotatable_bonds", "qed_weighted", "heavy_atoms", "aromatic_rings",
        "ro5_violations", "Target ChEMBL ID", "Target Name", "Target Organism"
    ]
    
    existing = [c for c in desiredOrder if c in chemblDataCombined.columns]
    extras = [c for c in chemblDataCombined.columns if c not in existing]
    chemblDataCombined = chemblDataCombined[existing + extras]

    outputPath = os.path.join(modelBuildingDataDir, f'{bacteriaName}Data_chEMBL_combined.csv')
    chemblDataCombined.to_csv(outputPath, index=False)
    print(f"Final combined data saved to: {outputPath} with shape {chemblDataCombined.shape}")
    print("\nProcessing complete.")
    
    return chemblDataCombined


def readCompoundsFileOptimized(filePath):
    """
    Optimized fallback parser for problematic CSV files.
    Uses list comprehension and minimal string operations.
    """
    with open(filePath, "r", encoding="utf-8", errors="replace") as f:
        # Read all at once - faster than line by line
        content = f.read()
    
    # Split into lines
    lines = content.split('\n')
    
    # Use csv module which is implemented in C and much faster
    import csv
    from io import StringIO
    
    reader = csv.reader(StringIO(content), delimiter=';', quotechar='"')
    rows = list(reader)
    
    if not rows:
        return pd.DataFrame()
    
    # Create DataFrame directly
    df = pd.DataFrame(rows[1:], columns=rows[0])
    return df




### Processing data files for `Bacillus_subtilis` bacteria 

In [49]:
bacteriaName = 'Bacillus_subtilis'  
bacteriaDataDir = os.path.join(dataDir, bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Bacillus_subtilis/Bacillus_subtilis_bioactivities.csv


/tmp/ipykernel_2421905/4160898751.py:31: DtypeWarning: Columns (31,45) have mixed types. Specify dtype option on import or set low_memory=False.
  dataFrame = pd.read_csv(


Dataset loaded successfully with Shape: (34560, 48)
Dataset shape after selecting columns: (34560, 48) → (34560, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_subtilisBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---
Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_subtilis_standard_types.png

--- Checking for NaN values ---
Smiles                   :    172 (  0.5%)
Standard Value           :  4,018 ( 11.6%)
Standard Units           :  5,020 ( 14.5%)
Standard Type            :      0 (  0.0%)
Standard Relation        :  4,034 ( 11.7%)
pChEMBL Value            : 34,360 ( 99.4%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :    172 (  0.5%)
AlogP                    :  2,275 (  6.6%)
Target ChEMBL ID         :      0 (  0.0%)
Target Name              :      0 (  0.0%)
Target Organism          :     

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL221435,CC1=C[C@H]2[C@@H](C(C)(C)O)CC[C@@](C)(O)[C@H]2CC1,C15H26O2,XOUUSQWJTXEKIT-PWNZVWSESA-N,InChI=1S/C15H26O2/c1-10-5-6-13-11(9-10)12(14(2...,238.369995,Small molecule,NaN,NaN,NaN,...,2.0,2.0,1.0,0.69,17.0,0.0,0.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
1,CHEMBL8,O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O,C17H18FN3O3,MYSWGUAQZAJSOK-UHFFFAOYSA-N,InChI=1S/C17H18FN3O3/c18-13-7-11-14(8-15(13)20...,331.350006,Small molecule,0.25,ug.mL-1,'=',...,5.0,2.0,3.0,0.89,24.0,2.0,0.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
2,CHEMBL606360,COc1ccc(C(=O)/C=C/c2cn(-c3ccccc3)nc2-c2ccc(Cl)...,C26H21ClN2O3,KEINEPLOMWOUNE-XNTDXEJSSA-N,InChI=1S/C26H21ClN2O3/c1-31-22-13-14-23(25(16-...,444.920013,Small molecule,100.00,ug.mL-1,'=',...,5.0,0.0,7.0,0.25,32.0,4.0,1.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
3,CHEMBL4089761,NC[C@H]1O[C@H](O[C@H]2[C@H](O)[C@@H](O[C@H]3O[...,C40H59FN10O14,VETOBWPFYDUOPP-FFKZLMLRSA-N,InChI=1S/C40H59FN10O14/c41-21-9-19-24(51(18-1-...,922.969971,Small molecule,1.50,ug.mL-1,'=',...,23.0,12.0,15.0,0.07,65.0,3.0,3.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
4,CHEMBL1241817,O=C(Cn1ncc2cc([N+](=O)[O-])ccc21)N/N=C/c1cccc(...,C16H12N6O5,DGJBCVGFPFQIOC-CAOOACKPSA-N,InChI=1S/C16H12N6O5/c23-16(19-17-8-11-2-1-3-13...,368.309998,Small molecule,16.00,ug.mL-1,'=',...,8.0,1.0,6.0,0.40,27.0,3.0,0.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34555,CHEMBL5574169,CC(C)(C)C(=O)Nc1nc(C(F)(F)F)c(-c2csc(Nc3cc(C(=...,C20H17F3N4O5S2,UCGIDVUDHVHHNH-UHFFFAOYSA-N,"InChI=1S/C20H17F3N4O5S2/c1-19(2,3)16(32)27-18-...",514.510010,NaN,32.00,ug.mL-1,'>',...,8.0,4.0,6.0,0.34,34.0,3.0,2.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
34556,CHEMBL5569723,CCC[C@@H]1Cc2cc3c(c(O)c2C(=O)O1)-c1c(c(OC)c2oc...,C46H54O20,JFJMNUYQKHCNOB-ZMBJTJKRSA-N,InChI=1S/C46H54O20/c1-6-7-19-10-18-11-20-30(38...,926.919983,NaN,1.00,ug.mL-1,'=',...,20.0,9.0,9.0,0.09,66.0,4.0,3.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis
34557,CHEMBL5405424,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC...,C57H103N13O9,TWJPADFFDBMMLU-UILVTTEASA-N,InChI=1S/C57H103N13O9/c1-34(2)28-44(49(62)71)6...,1114.530029,NaN,4.00,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL359,Bacillus subtilis,Bacillus subtilis
34558,CHEMBL5590188,CCCCCC(=O)N/N=c1\sc2ccccc2n1C,C14H19N3OS,DSMZDRMEDMHOSJ-PEZBUJJGSA-N,InChI=1S/C14H19N3OS/c1-3-4-5-10-13(18)15-16-14...,277.390015,NaN,512.00,ug.mL-1,'>',...,4.0,1.0,5.0,0.66,19.0,2.0,0.0,CHEMBL359,Bacillus subtilis,Bacillus subtilis


### Processing data files for `Bacillus_cereus` bacteria 

In [50]:
bacteriaName = 'Bacillus_cereus'  
bacteriaDataDir = os.path.join(dataDir, bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Bacillus_cereus/Bacillus_cereus_bioactivities.csv
Dataset loaded successfully with Shape: (7696, 48)
Dataset shape after selecting columns: (7696, 48) → (7696, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_cereusBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---
Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Bacillus_cereus_standard_types.png

--- Checking for NaN values ---
Smiles                   :     50 (  0.6%)
Standard Value           :  1,100 ( 14.3%)
Standard Units           :  1,194 ( 15.5%)
Standard Type            :      0 (  0.0%)
Standard Relation        :  1,098 ( 14.3%)
pChEMBL Value            :  7,687 ( 99.9%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :     50 (

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL372795,CN[C@@H]1[C@H](O[C@H]2[C@H](O[C@H]3[C@H](O)[C@...,C21H39N7O12,UCSJYZPVAKXKNQ-HZYVHMACSA-N,"InChI=1S/C21H39N7O12/c1-5-21(36,4-30)16(40-17-...",581.580017,Small molecule,20.530001,mm,'=',...,15.0,12.0,9.0,0.07,40.0,0.0,3.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
1,CHEMBL2238351,O=c1oc2ccccc2cc1-c1nnc(Sc2nc(Oc3cccc4cccnc34)n...,C32H23N9O4S,FKBMYQGGGKPKOB-UHFFFAOYSA-N,InChI=1S/C32H23N9O4S/c42-28-22(19-21-7-1-2-10-...,629.659973,Small molecule,50.000000,ug.mL-1,'=',...,14.0,0.0,7.0,0.21,46.0,7.0,3.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
2,CHEMBL4071700,CC1=CC2=C(CC(C)C(=O)C[C@H](C)O)C(=O)[C@](C)(O)...,C18H24O6,URYMFSUWSMOWTJ-UWFXQIJTSA-N,InChI=1S/C18H24O6/c1-9(15(20)6-10(2)19)5-13-12...,336.380005,Small molecule,NaN,NaN,NaN,...,6.0,3.0,5.0,0.70,24.0,0.0,0.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
3,CHEMBL4077237,CO[C@@H](/C=C/C=C(\C)[C@H](O)[C@H](C)C(=O)O[C@...,C29H46O8,UWGTWPNAWMDWHM-AHGVLRMZSA-N,"InChI=1S/C29H46O8/c1-16-13-21-12-11-18(3)29(6,...",522.679993,Small molecule,NaN,NaN,NaN,...,8.0,4.0,11.0,0.18,37.0,0.0,1.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
4,CHEMBL1276636,Cc1c(N2C(=O)C(Cl)=C(c3ccc[nH]3)C2=O)c(=O)n(-c2...,C19H15ClN4O3,PNPWLFUNSIHEEO-UHFFFAOYSA-N,InChI=1S/C19H15ClN4O3/c1-11-16(19(27)24(22(11)...,382.809998,Small molecule,NaN,NaN,NaN,...,5.0,1.0,3.0,0.71,27.0,3.0,0.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7691,CHEMBL5558032,Cn1cc(NC(=O)c2cc(NC(=O)c3ccc(/C=C/c4ccccc4)cc3...,C34H38N6O4,CSGUBISVSNYYLQ-MDZDMXLPSA-N,InChI=1S/C34H38N6O4/c1-38-24-29(21-30(38)33(42...,594.719971,NaN,70.000000,%,'=',...,7.0,3.0,11.0,0.18,44.0,4.0,1.0,CHEMBL613070,Bacillus cereus,Bacillus cereus
7692,CHEMBL5559579,CC(C)C[C@H](NC(=O)[C@H](CCCCN)NC(=O)CNC(=O)CNC...,C126H196N32O21,GTFVGDZKYFMXBQ-UIEXWCIESA-N,InChI=1S/C126H196N32O21/c1-72(2)53-97(118(171)...,2495.159912,NaN,32000.000000,nM,'>',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL613070,Bacillus cereus,Bacillus cereus
7693,CHEMBL5532125,CC(C)C[C@H](NC(=O)[C@H](CCCCN)NC(=O)CN)C(=O)N[...,C118H191N31O21,KMXQKIGASFXSSW-URGLQTMDSA-N,InChI=1S/C118H191N31O21/c1-67(2)50-89(110(162)...,2380.020020,NaN,4000.000000,nM,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL613070,Bacillus cereus,Bacillus cereus
7694,CHEMBL5556509,CC(C)C[C@H](NC(=O)[C@H](CCCCN)NC(=O)CNC(=O)CNC...,C120H195N31O21,MCZCEDIIGUEPSF-RLZKSDRCSA-N,InChI=1S/C120H195N31O21/c1-68(2)51-90(111(163)...,2408.080078,NaN,32000.000000,nM,'>',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL613070,Bacillus cereus,Bacillus cereus


### Processing data files for `Staphylococcus_aureus` bacteria 

In [51]:
bacteriaName = 'Staphylococcus_aureus'  
bacteriaDataDir = os.path.join(dataDir, bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Staphylococcus_aureus/Staphylococcus_aureus_bioactivities.csv


/tmp/ipykernel_2421905/4160898751.py:31: DtypeWarning: Columns (27,30,46) have mixed types. Specify dtype option on import or set low_memory=False.
  dataFrame = pd.read_csv(


Dataset loaded successfully with Shape: (232257, 48)
Dataset shape after selecting columns: (232257, 48) → (232257, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Staphylococcus_aureusBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---


/tmp/ipykernel_2421905/4160898751.py:95: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Staphylococcus_aureus_standard_types.png

--- Checking for NaN values ---
Smiles                   :  1,272 (  0.5%)
Standard Value           : 20,780 (  8.9%)
Standard Units           : 29,172 ( 12.6%)
Standard Type            :      0 (  0.0%)
Standard Relation        : 20,772 (  8.9%)
pChEMBL Value            : 230,915 ( 99.4%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :  1,180 (  0.5%)
AlogP                    : 26,831 ( 11.6%)
Target ChEMBL ID         :      0 (  0.0%)
Target Name              :      0 (  0.0%)
Target Organism          :      0 (  0.0%)

Number of unique compounds for Staphylococcus_aureus: 83473

--- Processing Compounds Data ---
Successfully read compounds file with pandas
Compounds data shape: (83354, 29)
Cleaned compounds data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Sta

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL507870,CCCCCCCCCCNCCN[C@@]1(C)C[C@H](O[C@H]2[C@H](Oc3...,C80H106Cl2N11O27P,ONUMZHGUFYIKPM-MXNFEBESSA-N,InChI=1S/C80H106Cl2N11O27P/c1-7-8-9-10-11-12-1...,1755.660034,Protein,3.300,NaN,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
1,CHEMBL1200628,CN[C@H](CC(C)C)C(=O)N[C@H]1C(=O)N[C@@H](CC(N)=...,C66H76Cl3N9O24,LCTORFDMHNKUSG-XTTLPDOESA-N,InChI=1S/C66H75Cl2N9O24.ClH/c1-23(2)12-34(71-5...,1485.729980,Small molecule,0.780,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
2,CHEMBL2164996,Cc1cc2c(cc1O)C1=C(C3=Cc4c(cc(O)c5cc(C)c(O)cc45...,C33H28O6,DFDVUOJRRAVNNF-UHFFFAOYSA-N,InChI=1S/C33H28O6/c1-14-7-18-16(10-24(14)34)17...,520.580017,Small molecule,25.000,ug.mL-1,'=',...,6.0,3.0,1.0,0.36,39.0,3.0,2.0,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
3,CHEMBL374478,CO[C@H]1/C=C/O[C@@]2(C)Oc3c(C)c(O)c4c(O)c(c(/C...,C43H58N4O12,JQXXHWHPUNPDRT-WLSIYKJHSA-N,InChI=1S/C43H58N4O12/c1-21-12-11-13-22(2)42(55...,822.950012,Small molecule,0.016,ug.mL-1,'=',...,15.0,6.0,4.0,0.11,59.0,2.0,3.0,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
4,CHEMBL376140,CN(C)c1cc(NC(=O)CNC(C)(C)C)c(O)c2c1C[C@H]1C[C@...,C29H39N5O8,FPZLLRFZJZRHSY-HJYUBDRYSA-N,"InChI=1S/C29H39N5O8/c1-28(2,3)31-11-17(35)32-1...",585.659973,Small molecule,0.125,ug.mL-1,'=',...,11.0,7.0,6.0,0.18,42.0,1.0,3.0,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232252,CHEMBL5570291,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC...,C60H100F3N13O10S,UKUVDXMWUHRYDL-GAQFJKPASA-N,InChI=1S/C60H100F3N13O10S/c1-37(2)30-48(53(68)...,1252.599976,NaN,32.000,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
232253,CHEMBL5574512,CC(C)C[C@H](N)CN(CC(=O)N[C@@H](CCCCN)C(=O)N[C@...,C57H102F3N13O10S,PCKQRAHXYIIWMC-KDXYNCTASA-N,InChI=1S/C57H102F3N13O10S/c1-34(2)27-40(64)32-...,1218.579956,NaN,32.000,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
232254,CHEMBL5405424,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC...,C57H103N13O9,TWJPADFFDBMMLU-UILVTTEASA-N,InChI=1S/C57H103N13O9/c1-34(2)28-44(49(62)71)6...,1114.530029,NaN,16.000,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus
232255,CHEMBL5314354,NaN,NaN,NaN,NaN,NaN,Protein,16.000,ug.mL-1,'=',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CHEMBL352,Staphylococcus aureus,Staphylococcus aureus


### Processing data files for `Enterococcus_faecalis` bacteria 

In [52]:
bacteriaName = 'Enterococcus_faecalis'  
bacteriaDataDir = os.path.join(dataDir, bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Enterococcus_faecalis/Enterococcus_faecalis_bioactivities.csv
Dataset loaded successfully with Shape: (32387, 48)
Dataset shape after selecting columns: (32387, 48) → (32387, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Enterococcus_faecalisBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---


/tmp/ipykernel_2421905/4160898751.py:95: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Enterococcus_faecalis_standard_types.png

--- Checking for NaN values ---
Smiles                   :    157 (  0.5%)
Standard Value           :  2,802 (  8.7%)
Standard Units           :  3,443 ( 10.6%)
Standard Type            :      0 (  0.0%)
Standard Relation        :  2,799 (  8.6%)
pChEMBL Value            : 32,336 ( 99.8%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecular Weight         :    157 (  0.5%)
AlogP                    :  5,076 ( 15.7%)
Target ChEMBL ID         :      0 (  0.0%)
Target Name              :      0 (  0.0%)
Target Organism          :      0 (  0.0%)

Number of unique compounds for Enterococcus_faecalis: 19447

--- Processing Compounds Data ---
Successfully read compounds file with pandas
Compounds data shape: (19428, 29)
Cleaned compounds data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Ente

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL1090696,CCOC(=O)C1=C(O)CC(c2ccccc2)N(C(O)CN2CCOCC2)C1c...,C26H32N2O5,JSNKZNVIDARGEO-UHFFFAOYSA-N,InChI=1S/C26H32N2O5/c1-2-33-26(31)24-22(29)17-...,452.549988,Small molecule,128.0,ug.mL-1,'>',...,7.0,2.0,7.0,0.62,33.0,2.0,0.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
1,CHEMBL1093337,CCOC(=O)C1=C(O)CC(c2ccccc2)N(C(O)CSc2ccc3ccccc...,C32H31NO4S,HBEMNXKELWUETR-UHFFFAOYSA-N,InChI=1S/C32H31NO4S/c1-2-37-32(36)30-28(34)20-...,525.669983,Small molecule,128.0,ug.mL-1,'>',...,6.0,2.0,8.0,0.19,38.0,4.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
2,CHEMBL1091116,CCOC(=O)C1=C(O)CC(c2ccccc2)N(C(O)C(C)n2ccc3ccc...,C31H32N2O4,QGDCMLVOKPHNPM-UHFFFAOYSA-N,InChI=1S/C31H32N2O4/c1-3-37-31(36)28-27(34)20-...,496.609985,Small molecule,128.0,ug.mL-1,'>',...,6.0,2.0,7.0,0.30,37.0,4.0,1.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
3,CHEMBL4098498,CCCCCCCCCCCCCCCOP(=O)(CCN(CCCN)CCCN)OC[C@H]1O[...,C32H62N5O8P,WRBRSZMBGQHTLE-JNVRDQITSA-N,InChI=1S/C32H62N5O8P/c1-2-3-4-5-6-7-8-9-10-11-...,675.849976,Small molecule,50.0,ug.mL-1,'=',...,12.0,5.0,28.0,0.06,46.0,1.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
4,CHEMBL4069782,CCCCCCCCCCCCCCCOP(=O)(CCN(CCN)CCN)OC[C@H]1O[C@...,C30H58N5O8P,BFFZRGRWYYMLOO-WZICAWFSSA-N,InChI=1S/C30H58N5O8P/c1-2-3-4-5-6-7-8-9-10-11-...,647.799988,Small molecule,12.5,ug.mL-1,'=',...,12.0,5.0,26.0,0.07,44.0,1.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32382,CHEMBL5567548,O=C(O)c1cc(Nc2nc(-c3sc(NC(=O)c4ccccc4)nc3C(F)(...,C22H12F6N4O3S2,QQLBDHNRPSORQL-UHFFFAOYSA-N,"InChI=1S/C22H12F6N4O3S2/c23-21(24,25)12-6-11(1...",558.489990,NaN,32.0,ug ml-1,'>',...,7.0,3.0,6.0,0.22,37.0,4.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
32383,CHEMBL5560856,CCCCCCc1c(-c2ccc(OCC(=O)CN[C@H](CCCNC(=N)N)C(N...,C40H58N10O10,XCXOBKSLLXGVCD-LOYHVIPDSA-N,InChI=1S/C40H58N10O10/c1-3-4-5-6-9-28-35(54)34...,838.960022,NaN,64.0,ug.mL-1,'>',...,14.0,11.0,29.0,0.03,60.0,3.0,3.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
32384,CHEMBL5559737,CCCCCCNCCCCOc1ccc(-c2oc3cc(OC)cc(O)c3c(=O)c2CC...,C42H66N2O6,FLDHDGFIFIMEDE-UHFFFAOYSA-N,InChI=1S/C42H66N2O6/c1-5-8-11-14-21-36-41(46)4...,695.000000,NaN,64.0,ug.mL-1,'=',...,8.0,3.0,29.0,0.06,50.0,3.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis
32385,CHEMBL5560242,CCCCCCc1c(-c2ccc(OCCCCN(C)C)cc2OCCCCN(C)C)oc2c...,C34H50N2O6,NSJASFJTZIBFCB-UHFFFAOYSA-N,InChI=1S/C34H50N2O6/c1-7-8-9-10-15-28-33(38)32...,582.780029,NaN,1.0,ug.mL-1,'=',...,8.0,1.0,19.0,0.16,42.0,3.0,2.0,CHEMBL362,Enterococcus faecalis,Enterococcus faecalis


### Processing data files for `Listeria_monocytogenes` bacteria 

In [53]:
bacteriaName = 'Listeria_monocytogenes'  
bacteriaDataDir = os.path.join(dataDir, bacteriaName)
processBacteriaData(bacteriaName)


--- Processing Bioassay Data ---
Reading ChEMBL CSV file: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Listeria_monocytogenes/Listeria_monocytogenes_bioactivities.csv
Dataset loaded successfully with Shape: (2701, 48)
Dataset shape after selecting columns: (2701, 48) → (2701, 12)
Cleaned bioassay data saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Listeria_monocytogenesBioassay_chEMBL_cleaned.csv

--- Analyzing Standard Types ---
Standard types plot saved to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/Listeria_monocytogenes_standard_types.png

--- Checking for NaN values ---
Smiles                   :     12 (  0.4%)
Standard Value           :    341 ( 12.6%)
Standard Units           :    405 ( 15.0%)
Standard Type            :      0 (  0.0%)
Standard Relation        :    341 ( 12.6%)
pChEMBL Value            :  2,685 ( 99.4%)
Molecule ChEMBL ID       :      0 (  0.0%)
Molecul

,molecule_chembl_id,smiles,molecular_formula,inchi_key,inchi,molecular_weight,type,standard_value,standard_units,standard_relation,...,hba,hbd,rotatable_bonds,qed_weighted,heavy_atoms,aromatic_rings,ro5_violations,Target ChEMBL ID,Target Name,Target Organism
0,CHEMBL5315124,C[C@H]1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)...,C36H42F2N6O9,SUIQUYDRLGGZOL-RCWTXCDDSA-N,InChI=1S/2C18H20FN3O4.H2O/c2*1-10-9-26-17-14-1...,740.760010,Small molecule,0.5,ug.mL-1,'=',...,6.0,1.0,2.0,0.87,26.0,2.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
1,CHEMBL1210162,O=C(O)c1ccc(/C=N/NC(=O)c2ccc(-c3nc4ccccc4[nH]3...,C22H16N4O3,HYEWSROTDZANKX-YDZHTSKRSA-N,InChI=1S/C22H16N4O3/c27-21(26-23-13-14-5-7-17(...,384.399994,Small molecule,400.0,ug.mL-1,'=',...,4.0,3.0,5.0,0.36,29.0,4.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
2,CHEMBL1236329,CC1=CC[C@@H]2C[C@H]1C2(C)C,C10H16,GRWFGVWFFZKLTI-RKDXNWHRSA-N,"InChI=1S/C10H16/c1-7-4-5-8-6-9(7)10(8,2)3/h4,8...",136.240005,Small molecule,0.0,%,'=',...,0.0,0.0,0.0,0.45,10.0,0.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
3,CHEMBL2269084,CCc1ccco1,C6H8O,HLPIHRDZBHXTFJ-UHFFFAOYSA-N,"InChI=1S/C6H8O/c1-2-6-4-3-5-7-6/h3-5H,2H2,1H3",96.129997,Small molecule,0.0,%,'=',...,1.0,0.0,1.0,0.52,7.0,1.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
4,CHEMBL545,CCO,C2H6O,LFQSCWFLJHTTHZ-UHFFFAOYSA-N,"InChI=1S/C2H6O/c1-2-3/h3H,2H2,1H3",46.070000,Small molecule,0.0,%,'=',...,1.0,1.0,0.0,0.41,3.0,0.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2696,CHEMBL4644199,FC(F)(F)c1ccc(NC(=S)N/N=C/c2ccc(Cl)cc2)cc1,C15H11ClF3N3S,BGKVILOXQJNMQS-AWQFTUOYSA-N,InChI=1S/C15H11ClF3N3S/c16-12-5-1-10(2-6-12)9-...,357.790009,Unknown,800.0,ug.mL-1,'=',...,2.0,2.0,3.0,0.48,23.0,2.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
2697,CHEMBL4632982,COc1ccc(/C=N/NC(=S)Nc2ccc(C(F)(F)F)cc2)cc1,C16H14F3N3OS,HPPOBRBATCMYHT-KEBDBYFISA-N,InChI=1S/C16H14F3N3OS/c1-23-14-8-2-11(3-9-14)1...,353.369995,Unknown,800.0,ug.mL-1,'=',...,3.0,2.0,4.0,0.49,24.0,2.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
2698,CHEMBL5171589,Clc1ccc2c(c1)C(NCCN1CCNCC1)=Nc1ccccc1O2,C19H21ClN4O,REXIYIBKBWXIKH-UHFFFAOYSA-N,InChI=1S/C19H21ClN4O/c20-14-5-6-17-15(13-14)19...,356.859985,NaN,1510.0,nM,'=',...,5.0,2.0,3.0,0.89,25.0,2.0,0.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes
2699,CHEMBL5208663,FC(F)(F)Sc1ccc2c(c1)[C@H]1[C@@H]3CC[C@@H](C3)[...,C19H17F6N3S,BYXLRDJYOSEADO-KROWVVRQSA-N,"InChI=1S/C19H17F6N3S/c20-18(21,22)17-12(7-26-2...",433.420013,NaN,4.0,ug.mL-1,'=',...,3.0,2.0,2.0,0.43,29.0,2.0,1.0,CHEMBL614974,Listeria monocytogenes,Listeria monocytogenes


### Combine the data set of all the bacteria into one large data set

In [54]:
# Explicitly list your CSV files
fileNames = [
    "Bacillus_subtilisData_chEMBL_combined.csv",
    "Bacillus_cereusData_chEMBL_combined.csv",
    "Staphylococcus_aureusData_chEMBL_combined.csv",
    "Enterococcus_faecalisData_chEMBL_combined.csv",
    "Listeria_monocytogenesData_chEMBL_combined.csv"
]

# Read and concatenate
dfList = []
for fileName in fileNames:
    filePath = os.path.join(modelBuildingDataDir, fileName)
    df = pd.read_csv(filePath)
    df["Virus"] = fileName.replace("_Data_chEMBL_combined.csv", "")
    print(f"Loaded {fileName:35s} → Shape: {df.shape}")
    dfList.append(df)

# Concatenate all
allVirusData_chEMBL = pd.concat(dfList, ignore_index=True)
print(f"\nFinal combined shape: {allVirusData_chEMBL.shape}")

# Save
outFile = os.path.join(modelBuildingDataDir, "AllBacteriaData_chEMBL_combined.csv")
allVirusData_chEMBL.to_csv(outFile, index=False)
print(f"Saved merged CSV to: {outFile}")

Loaded Bacillus_subtilisData_chEMBL_combined.csv → Shape: (34560, 25)
Loaded Bacillus_cereusData_chEMBL_combined.csv → Shape: (7696, 25)
Loaded Staphylococcus_aureusData_chEMBL_combined.csv → Shape: (232257, 25)
Loaded Enterococcus_faecalisData_chEMBL_combined.csv → Shape: (32387, 25)
Loaded Listeria_monocytogenesData_chEMBL_combined.csv → Shape: (2701, 25)

Final combined shape: (309601, 25)
Saved merged CSV to: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/modelBuildingData/AllBacteriaData_chEMBL_combined.csv
